# Metric Distribution Deep Dive

Explore the stability, out-of-distribution (OOD), and XRD metric distributions
for a given system. Supports overlaying multiple report files to see how
distributions shift across different configurations or reference packs.

Adjust the configuration cell below to point at your report(s).

In [ ]:
# ============================================================
# CONFIGURATION — edit these variables before running
# ============================================================
from pathlib import Path
from materials_discovery.common.io import workspace_root

WORKSPACE = workspace_root()

# Single report path (or extend REPORT_PATHS for overlay mode)
SYSTEM_NAME = "Al-Cu-Fe"
REPORT_PATH = WORKSPACE / "data" / "reports" / f"{SYSTEM_NAME.lower().replace('-', '_')}_report.json"

# For overlay mode: list multiple report paths with labels
# Example: [(path1, "Run A"), (path2, "Run B")]
OVERLAY_REPORTS: list[tuple[Path, str]] = [
    (REPORT_PATH, SYSTEM_NAME),
]

BINS = 20  # histogram bin count

print(f"System: {SYSTEM_NAME}")
for path, label in OVERLAY_REPORTS:
    print(f"  {label}: {path}  exists={path.exists()}")

In [ ]:
# ============================================================
# Load report entries (all overlay reports)
# ============================================================
import json

def load_entries(path: Path) -> list[dict]:
    """Load entries from a report JSON, gracefully returning [] if missing."""
    if not path.exists():
        print(f"[INFO] Report not found: {path}")
        print("       Run the pipeline first: mdisc report --config configs/systems/<system>.yaml")
        return []
    data = json.loads(path.read_text(encoding="utf-8"))
    return data.get("entries", [])


def extract_metric(entries: list[dict], metric: str) -> list[float]:
    """Extract a numeric metric from report entries, checking top-level and evidence."""
    values: list[float] = []
    for e in entries:
        val = e.get(metric)
        if val is None:
            val = e.get("evidence", {}).get(metric)
        if isinstance(val, (int, float)):
            values.append(float(val))
    return values


all_data: list[tuple[str, list[dict]]] = []
for path, label in OVERLAY_REPORTS:
    entries = load_entries(path)
    all_data.append((label, entries))
    if entries:
        print(f"{label}: {len(entries)} entries loaded")

# Use first available dataset as primary reference
primary_label, primary_entries = next(((l, e) for l, e in all_data if e), (SYSTEM_NAME, []))

In [ ]:
# ============================================================
# Histogram: hifi_score distribution
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

ALPHA = 0.6 if len(all_data) > 1 else 0.8
COLORS = ["#3498db", "#e67e22", "#2ecc71", "#9b59b6"]

fig, ax = plt.subplots(figsize=(8, 4))
has_data = False
for (label, entries), color in zip(all_data, COLORS):
    vals = extract_metric(entries, "hifi_score")
    if vals:
        ax.hist(vals, bins=BINS, alpha=ALPHA, label=label, color=color, edgecolor="white")
        has_data = True

if has_data:
    ax.set_xlabel("hifi_score (lower = better)")
    ax.set_ylabel("Count")
    ax.set_title("HiFi Score Distribution")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No hifi_score data available — run the pipeline first.")

In [ ]:
# ============================================================
# Histogram: stability_probability distribution
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4))
has_data = False
for (label, entries), color in zip(all_data, COLORS):
    vals = extract_metric(entries, "stability_probability")
    if vals:
        ax.hist(vals, bins=BINS, alpha=ALPHA, label=label, color=color, edgecolor="white")
        has_data = True

if has_data:
    ax.set_xlabel("stability_probability (higher = better)")
    ax.set_ylabel("Count")
    ax.set_title("Stability Probability Distribution")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No stability_probability data available.")

In [ ]:
# ============================================================
# Histogram: ood_score distribution
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4))
has_data = False
for (label, entries), color in zip(all_data, COLORS):
    vals = extract_metric(entries, "ood_score")
    if vals:
        ax.hist(vals, bins=BINS, alpha=ALPHA, label=label, color=color, edgecolor="white")
        has_data = True

if has_data:
    ax.set_xlabel("ood_score (lower = better in-distribution)")
    ax.set_ylabel("Count")
    ax.set_title("OOD Score Distribution")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No ood_score data available.")

In [ ]:
# ============================================================
# Scatter: xrd_confidence vs xrd_distinctiveness
# ============================================================
fig, ax = plt.subplots(figsize=(7, 5))
has_data = False
for (label, entries), color in zip(all_data, COLORS):
    conf = extract_metric(entries, "xrd_confidence")
    dist = extract_metric(entries, "xrd_distinctiveness")
    n = min(len(conf), len(dist))
    if n > 0:
        ax.scatter(conf[:n], dist[:n], alpha=0.5, label=label, color=color, s=20)
        has_data = True

if has_data:
    ax.set_xlabel("xrd_confidence")
    ax.set_ylabel("xrd_distinctiveness")
    ax.set_title("XRD Confidence vs Distinctiveness")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No XRD metric data available.")

In [ ]:
# ============================================================
# Summary statistics table
# ============================================================
import statistics

METRICS = [
    "hifi_score",
    "stability_probability",
    "ood_score",
    "xrd_confidence",
    "xrd_distinctiveness",
]

for label, entries in all_data:
    if not entries:
        continue
    print(f"\n=== Summary Statistics: {label} ===")
    print(f"{'Metric':<35} {'N':>5} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
    print("-" * 75)
    for metric in METRICS:
        vals = extract_metric(entries, metric)
        if not vals:
            print(f"{metric:<35} {'N/A':>5}")
            continue
        n = len(vals)
        mean = statistics.mean(vals)
        std = statistics.stdev(vals) if n > 1 else 0.0
        print(f"{metric:<35} {n:>5} {mean:>10.4f} {std:>10.4f} {min(vals):>10.4f} {max(vals):>10.4f}")

if not any(entries for _, entries in all_data):
    print("No data available. Run the pipeline first.")